In [28]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn import linear_model
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from math import sqrt
df = pd.read_excel("../../clean_coast_data.xlsx")

# using get_dummies on education column
encoded = pd.get_dummies(df.Area)
# Concatenate the dummies to original dataframe
merged_data = pd.concat([df, encoded], axis=1)
# dropping the original column which was not encoded
merged_data.drop(columns='Area', inplace=True)

merged_data.drop(columns=['coast_name','coast_desc','city','check station','key','coast_num','center'], inplace=True)
merged_data['Date test num'] = merged_data['Date test'].view(np.int64)


In [29]:
merged_data.columns

Index(['Distance from gaza', 'city rank', 'bathing_season', 'summer_vaction',
       'winter', 'river', 'military', 'nature', 'Date test', 'test_result',
       'test_result_num', 'month', 'year', 'city_jinny_ranking',
       'hifa_port_dis', 'sore_lengh', 'temperture ', 'rain', 'rain-1day',
       'rain-2days', 'rain-3days', 'temp-1day', 'temp-2days',
       'temp avg 3 days', 'Center', 'Eilat', 'North', 'South',
       'Date test num'],
      dtype='object')

In [30]:
merged_data.drop('Date test',axis=1,inplace=True)
merged_data.dropna(inplace=True)

In [31]:
northern_coasts = merged_data.loc[merged_data.North == 1]

X = northern_coasts.drop(columns=['test_result','test_result_num'])
y = northern_coasts[['test_result']]

In [33]:
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_regression
from sklearn.metrics import r2_score


f_reg_selector = SelectKBest(f_regression, k=10)
f_reg_selector.fit(X, y)
f_reg_support = f_reg_selector.get_support()
f_reg_feature = X.loc[:,f_reg_support].columns.to_list()

uniX = X.loc[:,f_reg_support]

X_train, X_test, y_train, y_test = train_test_split(uniX, y, shuffle=True, test_size = 0.2)
X_train_scaler = MinMaxScaler()
X_test_scaler = MinMaxScaler()
X_train_scaled = X_train_scaler.fit_transform(X_train)
X_test_scaled = X_test_scaler.fit_transform(X_test)

mlr = linear_model.LinearRegression()
mlr.fit(X_train_scaled,y_train.test_result)

prediction = mlr.predict(X_train_scaled)
resTrain =  pd.DataFrame(
    data={
        'Prediction':prediction, 
        'Actual': y_train.test_result.values    },
)
prediction = mlr.predict(X_test_scaled)
resTest = pd.DataFrame(
    data={
        'Prediction':prediction,
        'Actual': y_test.test_result.values    },
)

print('rmse train',sqrt(mean_squared_error(resTrain.Prediction, resTrain.Actual)))
print('std train',np.std(resTrain.Actual))
print('r2 train',r2_score(resTrain.Actual, resTrain.Prediction))

print('Sample')
print(resTrain.sample(10))
print()

print('rmse test',sqrt(mean_squared_error(resTest.Prediction, resTest.Actual)))
print('std test',np.std(resTest.Actual))
print('r2 test',r2_score(resTest.Actual, resTest.Prediction))

print('Sample')
print(resTest.sample(10))
print()

rmse train 0.9720925265642882
std train 1.0038289145470334
r2 train 0.06223114168613042
Sample
      Prediction  Actual
1386    3.377857       2
910     3.659969       2
1031    3.499931       4
4735    3.824785       3
559     3.464448       3
1148    3.252166       4
1397    3.895695       4
6092    3.764111       3
3352    3.967636       2
8067    3.918333       4

rmse test 0.9778098249496996
std test 1.0172173422071802
r2 test 0.07598019031368053
Sample
      Prediction  Actual
651     3.246103       5
204     3.422152       4
260     3.771075       3
668     3.880687       4
1920    3.665471       4
1361    3.802092       5
865     3.551136       2
709     3.780732       3
263     3.926671       4
1804    3.477866       5



c:\Users\liavc\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\utils\validation.py:72: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  return f(**kwargs)
c:\Users\liavc\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\feature_selection\_univariate_selection.py:302: RuntimeWarning: divide by zero encountered in true_divide
  corr /= X_norms
c:\Users\liavc\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\feature_selection\_univariate_selection.py:302: RuntimeWarning: invalid value encountered in true_divide
  corr /= X_norms
c:\Users\liavc\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\feature_selection\_univariate_selection.py:307: RuntimeWarning: invalid value encountered in true_divide
  F = corr ** 2 / (1 - corr ** 2) * degrees_of_freedom
